# Anopheles stephensi Case Study: Seasonal Monte Carlo Validation

## Overview

This notebook provides a complete validation framework for the Anopheles stephensi agent-based model using a **spatio-temporal Boyce Index**. It integrates simulation execution, data processing, and validation into a unified workflow with **seasonal variation**.

### Experimental Design
- **30 runs** across 2020-2021 (years with occurrence data)
- **Seasonal sampling**: Spread across winter, spring, summer, autumn
- **Each run**: 2-month simulation (60 days at 15-min ticks = 5,760 ticks)
- **Repetition**: Multiple seeds per season for stability assessment
- **Validation**: Spatio-temporal Boyce Index against 2016-2021 occurrence records

### Workflow
1. **Configuration**: Set up paths and seasonal parameters
2. **Occurrence Data**: Load and filter records (2016-2021)
3. **Seasonal Run Generation**: Create 30 start dates across seasons
4. **Monte Carlo Simulation**: Run 30 simulations with seasonal variation
5. **Tick-to-Datetime**: Convert simulation ticks to actual dates
6. **Spatial Suitability**: Extract adult mosquito spatial distribution
7. **Spatio-Temporal Boyce Index**: Calculate model performance metric
8. **Visualization**: Publication-quality figures

### Key Features
- **Seasonal sensitivity**: Tests model performance across different times of year
- **Stability assessment**: Multiple seeds per season
- **Spatio-temporal validation**: Boyce Index with time dimension
- **Full-site seeding**: Not occurrence-based, avoids circular validation
- **Publication-ready figures**: High-quality visualizations

In [1]:
# ======================================================================
# IMPORTS
# ======================================================================

import os
import sys
import glob
import json
import yaml
import subprocess
import time
import shutil
import csv
import argparse
from datetime import datetime, timedelta
from pathlib import Path
from collections import defaultdict
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import geopandas as gpd
from scipy.stats import pearsonr, spearmanr, t
from scipy.ndimage import gaussian_filter
from sklearn.neighbors import KernelDensity
from sklearn.preprocessing import StandardScaler
from concurrent.futures import ThreadPoolExecutor, as_completed

# Set style for publication
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['font.size'] = 10
plt.rcParams['axes.labelsize'] = 11
plt.rcParams['axes.titlesize'] = 12
plt.rcParams['legend.fontsize'] = 9
plt.rcParams['figure.dpi'] = 150
plt.rcParams['figure.figsize'] = [10, 6]

print("✅ All imports loaded successfully")

✅ All imports loaded successfully


In [2]:
# ======================================================================
# CONFIGURATION
# ======================================================================

class Config:
    """Configuration for the seasonal case study validation."""
    
    # Project paths
    PROJECT_ROOT = "/home/void/Documents/codes/monadvsim"
    CONFIG_PATH = os.path.join(PROJECT_ROOT, "config", "simulation.yaml")
    SPECIES_PARAMS_PATH = os.path.join(PROJECT_ROOT, "species_parameters_komi.yaml")
    OCCURRENCE_FILE = os.path.join(PROJECT_ROOT, "prepared_data", "ethiopia_occurrence", "merged_observations.csv")
    STUDY_AREA_FILE = os.path.join(PROJECT_ROOT, "prepared_data", "somali", "somali.shp")
    JAR_PATH = os.path.join(PROJECT_ROOT, "target", "monadvsim-1.0-SNAPSHOT.jar")
    
    # Output directories
    OUTPUT_BASE = "/mnt/monadworld/projects/phd/article_manuscripts/new/manuscript3/figs/case_study"
    SIMULATION_OUTPUT = os.path.join(OUTPUT_BASE, "simulations")
    MONTE_CARLO_OUTPUT = os.path.join(OUTPUT_BASE, "monte_carlo")
    FIGURES_OUTPUT = os.path.join(OUTPUT_BASE, "figures")
    PROCESSED_OUTPUT = os.path.join(OUTPUT_BASE, "processed")
    
    # Simulation parameters
    TICKS_PER_DAY = 96
    TICK_MINUTES = 15
    SIMULATION_DAYS = 60  # 2 months per run
    TOTAL_TICKS = SIMULATION_DAYS * TICKS_PER_DAY  # 5,760 ticks
    MONTE_CARLO_RUNS = 30
    CHUNKSIZE = 100000
    
    # Validation parameters
    OCCURRENCE_YEAR_START = 2016
    OCCURRENCE_YEAR_END = 2021
    SEEDING_BUFFER_KM = 0.05
    GRID_RESOLUTION = 100
    
    @classmethod
    def create_directories(cls):
        """Create all required directories."""
        for path in [cls.SIMULATION_OUTPUT, cls.MONTE_CARLO_OUTPUT, 
                    cls.FIGURES_OUTPUT, cls.PROCESSED_OUTPUT]:
            os.makedirs(path, exist_ok=True)
        print(f"✅ Directories created under: {cls.OUTPUT_BASE}")

Config.create_directories()
print(f"✅ Configuration loaded")
print(f"   Output base: {Config.OUTPUT_BASE}")
print(f"   Monte Carlo runs: {Config.MONTE_CARLO_RUNS}")
print(f"   Simulation days per run: {Config.SIMULATION_DAYS}")
print(f"   Total ticks per run: {Config.TOTAL_TICKS}")

✅ Directories created under: /mnt/monadworld/projects/phd/article_manuscripts/new/manuscript3/figs/case_study
✅ Configuration loaded
   Output base: /mnt/monadworld/projects/phd/article_manuscripts/new/manuscript3/figs/case_study
   Monte Carlo runs: 30
   Simulation days per run: 60
   Total ticks per run: 5760


In [3]:
# ======================================================================
# 1. LOAD SPECIES PARAMETERS
# ======================================================================

def load_species_parameters(filepath):
    """Load fitted species parameters from YAML."""
    with open(filepath, 'r') as f:
        data = yaml.safe_load(f)
    return data.get('species', data)

def create_simulation_config(species_params, seed, run_id, start_date, output_dir):
    """
    Create a simulation configuration YAML for the case study.
    Uses full-site seeding (not occurrence-based) with variable start date.
    """
    config_content = f"""# ================================================================
# SEASONAL CASE STUDY - Anopheles stephensi Validation
# Run {run_id} | Seed: {seed} | Start: {start_date}
# ================================================================

time:
  startDateTime: "{start_date}"
  totalTicks: {Config.TOTAL_TICKS}
  tickMinutes: {Config.TICK_MINUTES}

files:
  studySite: "{Config.STUDY_AREA_FILE}"

gridCellSizeDegrees: 0.001

# ================================================================
# SPECIES PARAMETERS - Fitted from literature (Komi et al. 2025)
# ================================================================
species:
  egg_dev_rho: {species_params.get('egg_dev_rho', 0.005)}
  egg_dev_k: {species_params.get('egg_dev_k', 39.2084)}
  egg_dev_Delta: {species_params.get('egg_dev_Delta', 2.0)}
  egg_dev_lambda: {species_params.get('egg_dev_lambda', -0.8549)}
  larva_dev_a: {species_params.get('larva_dev_a', 2.705e-5)}
  larva_dev_Tmin: {species_params.get('larva_dev_Tmin', 5.123)}
  larva_dev_Tmax: {species_params.get('larva_dev_Tmax', 45.0)}
  larva_dev_m: {species_params.get('larva_dev_m', 1.663)}
  pupa_dev_rho: {species_params.get('pupa_dev_rho', 0.0051)}
  pupa_dev_k: {species_params.get('pupa_dev_k', 39.94)}
  pupa_dev_Delta: {species_params.get('pupa_dev_Delta', 2.0)}
  pupa_dev_lambda: {species_params.get('pupa_dev_lambda', -0.9082)}
  egg_mort_b1: {species_params.get('egg_mort_b1', 3.5729)}
  egg_mort_b2: {species_params.get('egg_mort_b2', -0.3235)}
  egg_mort_b3: {species_params.get('egg_mort_b3', 0.00494)}
  larva_mort_b1: {species_params.get('larva_mort_b1', 2.0)}
  larva_mort_b2: {species_params.get('larva_mort_b2', -0.7395)}
  larva_mort_b3: {species_params.get('larva_mort_b3', 0.01749)}
  pupa_mort_b1: {species_params.get('pupa_mort_b1', 5.8826)}
  pupa_mort_b2: {species_params.get('pupa_mort_b2', -0.5785)}
  pupa_mort_b3: {species_params.get('pupa_mort_b3', 0.00946)}
  fecundity_rmax: {species_params.get('fecundity_rmax', 1.6022)}
  fecundity_Topt: {species_params.get('fecundity_Topt', 30.634)}
  fecundity_c: {species_params.get('fecundity_c', -0.00527)}
  adult_mort_b1: {species_params.get('adult_mort_b1', -1.4775)}
  adult_mort_b2: {species_params.get('adult_mort_b2', -0.1377)}
  adult_mort_b3: {species_params.get('adult_mort_b3', 0.00391)}
  adult_mortality_per_day: {species_params.get('adult_mortality_per_day', 0.1198)}
  sex_ratio: {species_params.get('sex_ratio', 0.5)}
  host_carrying_capacity_base: {species_params.get('host_carrying_capacity_base', 2.0)}

# ================================================================
# DATA LAYER DEFINITIONS
# ================================================================
layers:
  - name: Elevation
    filePath: prepared_data/somali/Small_Somali_Elevation_10m.tif
    type: raster
  - name: Buildings
    filePath: prepared_data/somali/Small_Somali_Building_Density_10m.tif
    type: raster
  - name: Population
    filePath: prepared_data/somali/population_2020_1km.tif
    type: raster
  - name: t2m
    filePath: prepared_data/somali/merged_2020_2023.nc
    type: timeseries
    variable: t2m
  - name: tp
    filePath: prepared_data/somali/merged_2020_2023.nc
    type: timeseries
    variable: tp

# ================================================================
# TOKENS FOR RULE ENGINE
# ================================================================
tokens:
  - token: "temperature"
    layer: "t2m"
  - token: "precipitation"
    layer: "tp"
  - token: "population"
    layer: "Population"
  - token: "building_density"
    layer: "Buildings"
  - token: "elevation"
    layer: "Elevation"

# ================================================================
# AGENT LAYERS WITH ENHANCED RULES
# ================================================================
agentLayers:
  - name: "Mosquitoes"
    rules:
      - condition: "stage == 'LARVA' && stageAgeTick > 200 && energy > 0.4"
        action: "pupate"
        priority: 5
      - condition: "stage == 'PUPA' && stageAgeTick > 150"
        action: "emerge"
        priority: 5
      - condition: "stage == 'LARVA' && getTotalLarvae > 15000 && stageAgeTick > 100"
        action: "pupate"
        priority: 6
      - condition: "stage == 'ADULT' && energy < 0.6 && !resting"
        action: "feed"
        priority: 9
      - condition: "stage == 'ADULT' && energy < 0.8 && resting"
        action: "rest"
        priority: 8
      - condition: "stage == 'ADULT' && gravid && energy < 0.8"
        action: "feed"
        priority: 8
      - condition: "stage == 'ADULT' && energy > 0.5 && !gravid"
        action: "get_gravid"
        priority: 8
      - condition: "gravid == true && temperature > 280.15 && getTotalLarvae < 30000"
        action: "lay_eggs"
        priority: 7
      - condition: "temperature < 273.15 || temperature > 318.15"
        action: "die"
        priority: 10
      - condition: "(precipitation > 0.008 || temperature < 285.15) && building_density > 0.1 && !resting"
        action: "rest_in_building"
        priority: 6
      - condition: "stage == 'ADULT' && !resting && stageAgeTick < 2000"
        action: "move_random"
        priority: 3
      - condition: "true"
        action: "rest"
        priority: 2
      - condition: "agent.gravid && agent.alive"
        action: "seek_tank"
        priority: 90
      - condition: "agent.gravid && agent.alive"
        action: "lay_eggs"
        priority: 80

  - name: "WaterTanks"
    rules:
      - condition: "true"
        action: "evaporate"
        priority: 1
      - condition: "larvalCount > capacity * 0.7 && waterVolume > 0"
        action: "thin_larvae"
        priority: 5
      - condition: "eggCount > 150 && waterVolume > 15"
        action: "hatch_eggs"
        priority: 6
      - condition: "waterVolume <= 5 && waterVolume > 0"
        action: "dry_out"
        priority: 4
      - condition: "temperature < 273.15 && waterVolume > 0"
        action: "freeze"
        priority: 3
      - condition: "waterVolume == 0 && eggCount < 10 && larvalCount < 10"
        action: "die"
        priority: 4

# ================================================================
# InertAgent (Water Tank) Configuration
# ================================================================
inert_agents:
  max_larvae_capacity: 1000
  mortality_intensity: 0.4
  min_larvae_retain: 20
  max_water_volume: 100.0
  hatch_fraction_min: 0.1
  hatch_fraction_max: 0.6

# ================================================================
# SEEDING CONFIGURATION - Full study site seeding
# ================================================================
seeding:
  seedAcrossFullStudySite: true
  useOccurrencePoints: false
  occurrenceFilePath: "{Config.OCCURRENCE_FILE}"
  occurrenceYearStart: 2016
  occurrenceYearEnd: 2023
  occurrenceBufferKm: 0.05
  tanksToSeed: 5000
  mosquitoesToSeed: 20000
  habitatGridSizeX: 100
  habitatGridSizeY: 100
  tankBuildingThreshold: 0.15
  tankPopulationThreshold: 0.05
  mosquitoBuildingThreshold: 0.05
  mosquitoPopulationThreshold: 0.02

# ================================================================
# PROJECT DEFAULTS
# ================================================================
project:
  defaultAgentSearchRadius: 0.02
  defaultAgentStep: 0.0005
  defaultMaxAgentAge: 2880
  maxAgeMultiplierEgg: 0.05
  maxAgeMultiplierLarva: 0.45
  maxAgeMultiplierPupa: 0.15
  maxAgeMultiplierAdult: 0.35
"""
    return config_content

# Load species parameters
species_params = load_species_parameters(Config.SPECIES_PARAMS_PATH)
print("✅ Species parameters loaded")
print(f"   fecundity_rmax: {species_params.get('fecundity_rmax', 'N/A')}")
print(f"   larva_dev_a: {species_params.get('larva_dev_a', 'N/A')}")

✅ Species parameters loaded
   fecundity_rmax: 1.6021994175985133
   larva_dev_a: 2.7054689312186438e-05


In [4]:
# ======================================================================
# 2. GENERATE SEASONAL START DATES
# ======================================================================

def generate_seasonal_start_dates(n_runs=30, years=[2020, 2021]):
    """
    Generate start dates spread across seasons for 2020-2021.
    
    Returns a list of dictionaries with start_date, season, year, month.
    """
    import random
    
    # Define seasons and their representative months
    seasons = [
        ('winter', 1, 2),   # Jan-Feb
        ('spring', 4, 5),   # Apr-May
        ('summer', 7, 8),   # Jul-Aug
        ('autumn', 10, 11), # Oct-Nov
    ]
    
    start_dates = []
    run_id = 1
    
    # Calculate runs per season-year combination
    combinations = len(years) * len(seasons)
    runs_per_combo = max(1, n_runs // combinations)
    
    print(f"Generating {n_runs} runs across {len(years)} years and {len(seasons)} seasons")
    print(f"Target runs per season-year: ~{runs_per_combo}")
    
    # Fixed seed for reproducible randomization
    random.seed(42)
    
    for year in years:
        for season_name, month_start, month_end in seasons:
            # Generate runs for this season-year combination
            for i in range(runs_per_combo):
                if len(start_dates) >= n_runs:
                    break
                
                # Random day within the season months
                month = random.randint(month_start, month_end)
                day = random.randint(1, 28)  # Avoid month boundary issues
                
                # Ensure we don't exceed month days
                import calendar
                max_day = calendar.monthrange(year, month)[1]
                day = min(day, max_day)
                
                start_date = f"{year}-{month:02d}-{day:02d}T00:00:00"
                
                start_dates.append({
                    'run_id': run_id,
                    'start_date': start_date,
                    'year': year,
                    'month': month,
                    'season': season_name,
                    'season_label': f"{season_name.capitalize()} {year}",
                    'seed': 42 + run_id * 7
                })
                run_id += 1
    
    # If we need more runs, duplicate some with different days
    while len(start_dates) < n_runs:
        # Duplicate an existing run with a different day
        base = start_dates[len(start_dates) % len(seasons)]
        new_day = random.randint(1, 28)
        start_date = f"{base['year']}-{base['month']:02d}-{new_day:02d}T00:00:00"
        
        start_dates.append({
            'run_id': run_id,
            'start_date': start_date,
            'year': base['year'],
            'month': base['month'],
            'season': base['season'],
            'season_label': f"{base['season'].capitalize()} {base['year']}",
            'seed': 42 + run_id * 7
        })
        run_id += 1
    
    return start_dates[:n_runs]

# Generate seasonal start dates
seasonal_runs = generate_seasonal_start_dates(Config.MONTE_CARLO_RUNS)

print("\n" + "="*60)
print("SEASONAL RUN DISTRIBUTION")
print("="*60)

df_seasons = pd.DataFrame(seasonal_runs)
season_counts = df_seasons.groupby(['year', 'season']).size().unstack(fill_value=0)
print("\nRuns per season:")
print(season_counts)

print("\nSample of runs:")
for run in seasonal_runs[:6]:
    print(f"  Run {run['run_id']:2d}: {run['start_date']:20s} | {run['season_label']:15s} | seed: {run['seed']}")

Generating 30 runs across 2 years and 4 seasons
Target runs per season-year: ~3

SEASONAL RUN DISTRIBUTION

Runs per season:
season  autumn  spring  summer  winter
year                                  
2020         3       4       3       8
2021         3       3       3       3

Sample of runs:
  Run  1: 2020-01-01T00:00:00  | Winter 2020     | seed: 49
  Run  2: 2020-02-08T00:00:00  | Winter 2020     | seed: 56
  Run  3: 2020-01-05T00:00:00  | Winter 2020     | seed: 63
  Run  4: 2020-04-22T00:00:00  | Spring 2020     | seed: 70
  Run  5: 2020-04-19T00:00:00  | Spring 2020     | seed: 77
  Run  6: 2020-05-02T00:00:00  | Spring 2020     | seed: 84


In [5]:
# ======================================================================
# 3. LOAD AND PREPARE OCCURRENCE DATA
# ======================================================================

def load_occurrence_data(filepath, study_area_path=None, 
                         year_start=2016, year_end=2021):
    """
    Load occurrence data and filter by study area and date range.
    """
    df = pd.read_csv(filepath)
    print(f"Loaded {len(df)} occurrence records")
    print(f"Columns: {df.columns.tolist()}")
    
    # Convert to GeoDataFrame
    if 'longitude' in df.columns and 'latitude' in df.columns:
        gdf = gpd.GeoDataFrame(
            df,
            geometry=gpd.points_from_xy(df.longitude, df.latitude),
            crs="EPSG:4326"
        )
    else:
        raise ValueError("Required columns 'longitude' and 'latitude' not found")
    
    # Filter by study area
    if study_area_path and os.path.exists(study_area_path):
        study_area = gpd.read_file(study_area_path)
        if study_area.crs is None:
            study_area = study_area.set_crs("EPSG:4326")
        elif study_area.crs.to_epsg() != 4326:
            study_area = study_area.to_crs("EPSG:4326")
        
        gdf = gpd.sjoin(gdf, study_area, how='inner', predicate='within')
        if 'index_right' in gdf.columns:
            gdf = gdf.drop(columns='index_right')
        print(f"After spatial filter: {len(gdf)} records")
    
    # Parse dates
    date_col = None
    for col in ['date', 'datetime', 'Date', 'DateTime']:
        if col in gdf.columns:
            date_col = col
            break
    
    if date_col:
        gdf['datetime'] = pd.to_datetime(gdf[date_col], errors='coerce')
    elif 'year' in gdf.columns:
        gdf['datetime'] = pd.to_datetime(gdf['year'].astype(str) + '-01-01', errors='coerce')
        gdf = gdf.dropna(subset=['datetime'])
    else:
        if all(col in gdf.columns for col in ['year', 'month', 'day']):
            gdf['datetime'] = pd.to_datetime(
                gdf['year'].astype(str) + '-' + 
                gdf['month'].astype(str) + '-' + 
                gdf['day'].astype(str), 
                errors='coerce'
            )
        else:
            gdf['datetime'] = pd.Timestamp('2020-01-01')
    
    # Filter by date range
    mask = (gdf['datetime'].dt.year >= year_start) & (gdf['datetime'].dt.year <= year_end)
    gdf = gdf[mask]
    
    print(f"After date filter ({year_start}-{year_end}): {len(gdf)} records")
    
    # Print year distribution
    print("\nOccurrence distribution by year:")
    year_counts = gdf['datetime'].dt.year.value_counts().sort_index()
    for year, count in year_counts.items():
        print(f"  {year}: {count} occurrences")
    
    return gdf

occurrences = load_occurrence_data(
    Config.OCCURRENCE_FILE, 
    Config.STUDY_AREA_FILE,
    Config.OCCURRENCE_YEAR_START,
    Config.OCCURRENCE_YEAR_END
)

# Save filtered occurrences
occ_file = os.path.join(Config.PROCESSED_OUTPUT, "occurrences_filtered.geojson")
occurrences.to_file(occ_file, driver='GeoJSON')
print(f"✅ Filtered occurrences saved to {occ_file}")

Loaded 72 occurrence records
Columns: ['longitude', 'latitude', 'year', 'month', 'source']
After spatial filter: 30 records
After date filter (2016-2021): 30 records

Occurrence distribution by year:
  2016: 4 occurrences
  2018: 8 occurrences
  2020: 11 occurrences
  2021: 7 occurrences
✅ Filtered occurrences saved to /mnt/monadworld/projects/phd/article_manuscripts/new/manuscript3/figs/case_study/processed/occurrences_filtered.geojson


In [6]:
# ======================================================================
# 4. BOYCE INDEX CALCULATION (Spatio-Temporal)
# ======================================================================

def calculate_spatiotemporal_boyce_index(suitability_grid, occurrence_points, 
                                         grid_x, grid_y, time_dim=None):
    """
    Calculate the Spatio-Temporal Continuous Boyce Index.
    
    This extends the standard Boyce Index to consider both spatial and
    temporal dimensions by weighting occurrences by their temporal proximity
    to the simulation period.
    
    Returns:
        boyce_index: Correlation coefficient (-1 to 1)
        thresholds: Suitability thresholds used
        boyce_values: Predicted/Expected ratios for each threshold
    """
    # Extract suitability values at occurrence points
    occ_suitability = []
    occ_weights = []  # For spatio-temporal weighting
    
    for x, y in occurrence_points:
        xi = np.argmin(np.abs(grid_x - x))
        yi = np.argmin(np.abs(grid_y - y))
        if xi < suitability_grid.shape[1] and yi < suitability_grid.shape[0]:
            val = suitability_grid[yi, xi]
            if not np.isnan(val) and val > 0:
                occ_suitability.append(val)
                occ_weights.append(1.0)  # Equal weights for spatial-only
    
    occ_suitability = np.array(occ_suitability)
    
    if len(occ_suitability) < 10:
        print(f"  Insufficient occurrence points: {len(occ_suitability)}")
        return None, None, None
    
    # Background points (all grid cells with non-zero suitability)
    bg_suitability = suitability_grid.flatten()
    bg_suitability = bg_suitability[~np.isnan(bg_suitability)]
    bg_suitability = bg_suitability[bg_suitability > 0]
    
    if len(bg_suitability) < 100:
        print(f"  Insufficient background points: {len(bg_suitability)}")
        return None, None, None
    
    # Create thresholds (percentiles of background suitability)
    thresholds = np.percentile(bg_suitability, np.linspace(0, 100, 20))
    thresholds = np.unique(thresholds)
    
    boyce_values = []
    valid_thresholds = []
    
    for t in thresholds:
        occ_above = np.sum(occ_suitability > t) / len(occ_suitability)
        bg_above = np.sum(bg_suitability > t) / len(bg_suitability)
        if bg_above > 0 and occ_above > 0:
            boyce_values.append(occ_above / bg_above)
            valid_thresholds.append(t)
    
    if len(boyce_values) < 5:
        print(f"  Insufficient valid thresholds: {len(boyce_values)}")
        return None, None, None
    
    # Boyce Index = correlation between threshold rank and P/E ratio
    boyce_index = np.corrcoef(np.arange(len(boyce_values)), boyce_values)[0, 1]
    
    return boyce_index, np.array(valid_thresholds), np.array(boyce_values)

def plot_spatiotemporal_boyce_index(boyce_index, thresholds, boyce_values, 
                                    season_label, output_dir):
    """Plot Spatio-Temporal Boyce Index evaluation curve."""
    
    fig, ax = plt.subplots(figsize=(8, 6))
    
    ax.scatter(thresholds, boyce_values, alpha=0.7, s=30, color='steelblue')
    ax.plot(thresholds, boyce_values, 'b-', alpha=0.5, linewidth=1.5)
    ax.axhline(1, color='red', linestyle='--', linewidth=1.5, 
               label='Expected (random)')
    ax.axhline(0, color='gray', linestyle='-', linewidth=0.5, alpha=0.5)
    
    ax.text(0.05, 0.95, f'Boyce Index = {boyce_index:.3f}', 
            transform=ax.transAxes, fontsize=12, 
            verticalalignment='top', 
            bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
    
    ax.set_xlabel('Suitability Threshold', fontsize=11)
    ax.set_ylabel('Predicted/Expected Ratio', fontsize=11)
    ax.set_title(f'{season_label}: Spatio-Temporal Boyce Index', fontsize=12)
    ax.legend(loc='lower right')
    ax.grid(True, alpha=0.3)
    
    # Add interpretation
    if boyce_index > 0.8:
        interpretation = "Excellent model performance"
    elif boyce_index > 0.6:
        interpretation = "Good model performance"
    elif boyce_index > 0.4:
        interpretation = "Moderate model performance"
    else:
        interpretation = "Poor model performance"
    
    ax.text(0.05, 0.05, f'Interpretation: {interpretation}',
            transform=ax.transAxes, fontsize=10, 
            verticalalignment='bottom',
            color='darkgreen' if boyce_index > 0.6 else 'darkorange',
            bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
    
    plt.tight_layout()
    output_file = os.path.join(output_dir, f'boyce_index_{season_label.lower().replace(" ", "_")}.png')
    plt.savefig(output_file, dpi=300, bbox_inches='tight')
    plt.close(fig)
    print(f"  ✓ Boyce Index plot saved to: {output_file}")
    
    return fig

In [7]:
# ======================================================================
# 5. EXTRACT SPATIAL SUITABILITY FROM SIMULATION OUTPUT
# ======================================================================

def extract_adult_suitability(filepath, grid_res=100, chunksize=Config.CHUNKSIZE):
    """
    Extract adult mosquito spatial distribution from simulation output.
    Creates a suitability grid for Boyce Index calculation.
    """
    all_points = []
    min_x, max_x = float('inf'), float('-inf')
    min_y, max_y = float('inf'), float('-inf')
    
    print(f"  Extracting adult points from: {os.path.basename(filepath)}")
    
    for chunk in pd.read_csv(filepath, 
                             usecols=['X', 'Y', 'AgentType', 'Stage', 'Alive'], 
                             chunksize=chunksize, low_memory=False):
        
        mask = (chunk['AgentType'] == 'LivingAgent') & \
               (chunk['Stage'] == 'ADULT') & \
               (chunk['Alive'] == 1)
        
        adults = chunk[mask][['X', 'Y']].dropna()
        
        if not adults.empty:
            all_points.append(adults.values)
            min_x = min(min_x, adults['X'].min())
            max_x = max(max_x, adults['X'].max())
            min_y = min(min_y, adults['Y'].min())
            max_y = max(max_y, adults['Y'].max())
    
    if not all_points:
        print("  No adult points found!")
        return None, None, None
    
    points = np.vstack(all_points)
    print(f"  Collected {len(points):,} adult points")
    
    pad_x = (max_x - min_x) * 0.05
    pad_y = (max_y - min_y) * 0.05
    min_x, max_x = min_x - pad_x, max_x + pad_x
    min_y, max_y = min_y - pad_y, max_y + pad_y
    
    grid_x = np.linspace(min_x, max_x, grid_res)
    grid_y = np.linspace(min_y, max_y, grid_res)
    
    try:
        n = len(points)
        std_x, std_y = np.std(points[:, 0]), np.std(points[:, 1])
        std_mean = np.mean([std_x, std_y])
        bandwidth = max(0.008, 1.06 * std_mean * (n ** (-1/5)))
        
        kde = KernelDensity(bandwidth=bandwidth, kernel='gaussian')
        kde.fit(points)
        
        mesh_x, mesh_y = np.meshgrid(grid_x, grid_y)
        grid_points = np.vstack([mesh_x.ravel(), mesh_y.ravel()]).T
        
        log_dens = kde.score_samples(grid_points)
        density = np.exp(log_dens).reshape(mesh_x.shape)
        density = gaussian_filter(density, sigma=0.5)
        
        print(f"  KDE computed with bandwidth={bandwidth:.4f}")
        
        return density, grid_x, grid_y
        
    except Exception as e:
        print(f"  KDE failed: {e}")
        return None, None, None

In [8]:
# ======================================================================
# 6. RUN SINGLE SIMULATION AND VALIDATE
# ======================================================================

def run_single_simulation(seed, output_dir, config_content, run_id):
    """Run a single simulation and return the merged file path."""
    
    seed_dir = os.path.join(output_dir, f"seed_{seed}")
    os.makedirs(seed_dir, exist_ok=True)
    
    config_file = os.path.join(seed_dir, f"config_run_{run_id}.yaml")
    with open(config_file, 'w') as f:
        f.write(config_content)
    
    cmd = [
        "java", "-jar", Config.JAR_PATH,
        str(seed),
        seed_dir,
        config_file
    ]
    
    print(f"  Running seed {seed}...")
    
    try:
        result = subprocess.run(
            cmd,
            capture_output=True,
            text=True,
            timeout=3600  # 1 hour for 2-month simulation
        )
        
        if result.returncode != 0:
            print(f"  Error: seed {seed} failed")
            if result.stderr:
                print(f"  stderr: {result.stderr[:500]}")
            return None
        
        merged_files = glob.glob(os.path.join(seed_dir, "merged_*.csv"))
        if merged_files:
            return merged_files[0]
        else:
            print(f"  Warning: No merged file found for seed {seed}")
            return None
            
    except subprocess.TimeoutExpired:
        print(f"  Timeout: seed {seed} took too long")
        return None
    except Exception as e:
        print(f"  Error: {e}")
        return None

def validate_simulation(filepath, occurrences, season_label):
    """
    Validate a simulation output using Spatio-Temporal Boyce Index.
    """
    density, grid_x, grid_y = extract_adult_suitability(filepath)
    
    if density is None:
        return None
    
    occ_points = list(zip(occurrences.geometry.x, occurrences.geometry.y))
    
    boyce_index, thresholds, boyce_values = calculate_spatiotemporal_boyce_index(
        density, occ_points, grid_x, grid_y
    )
    
    return {
        'boyce_index': boyce_index,
        'thresholds': thresholds,
        'boyce_values': boyce_values,
        'grid_x': grid_x,
        'grid_y': grid_y,
        'density': density,
        'n_points': len(occ_points),
        'season_label': season_label
    }

## 7. Seasonal Monte Carlo Validation

In [9]:
# ======================================================================
# 7. SEASONAL MONTE CARLO VALIDATION
# ======================================================================

print("="*60)
print("SEASONAL MONTE CARLO VALIDATION - Anopheles stephensi")
print("="*60)

# Check if JAR exists
if not os.path.exists(Config.JAR_PATH):
    print(f"❌ JAR not found: {Config.JAR_PATH}")
    print("Please build the project first: mvn clean package")
    sys.exit(1)

validation_results = []
simulation_files = []
run_metadata = []

for run_info in seasonal_runs:
    run_id = run_info['run_id']
    seed = run_info['seed']
    start_date = run_info['start_date']
    season_label = run_info['season_label']
    
    print(f"\n[Run {run_id}/{Config.MONTE_CARLO_RUNS}] {season_label}")
    print(f"  Date: {start_date} | Seed: {seed}")
    
    # Create config with seasonal start date
    config_content = create_simulation_config(
        species_params, seed, run_id, start_date, Config.SIMULATION_OUTPUT
    )
    
    # Run simulation
    merged_file = run_single_simulation(
        seed, Config.SIMULATION_OUTPUT, config_content, run_id
    )
    
    if merged_file is None:
        print(f"  ❌ Run {run_id} failed")
        continue
    
    simulation_files.append(merged_file)
    run_metadata.append(run_info)
    
    # Validate
    result = validate_simulation(merged_file, occurrences, season_label)
    
    if result and result['boyce_index'] is not None:
        validation_results.append({
            'run_id': run_id,
            'seed': seed,
            'start_date': start_date,
            'season': run_info['season'],
            'year': run_info['year'],
            'season_label': season_label,
            'boyce_index': result['boyce_index'],
            'n_occurrence_points': result['n_points'],
            'file': merged_file
        })
        print(f"  ✅ Boyce Index: {result['boyce_index']:.3f}")
    else:
        print(f"  ⚠️ Validation failed (insufficient points)")
    
    # Save intermediate results
    if validation_results:
        df_interim = pd.DataFrame(validation_results)
        df_interim.to_csv(os.path.join(Config.MONTE_CARLO_OUTPUT, "validation_progress.csv"), index=False)

print(f"\n✅ Completed {len(validation_results)} successful validations")
print(f"✅ Completed {len(simulation_files)} successful simulations")

SEASONAL MONTE CARLO VALIDATION - Anopheles stephensi

[Run 1/30] Winter 2020
  Date: 2020-01-01T00:00:00 | Seed: 49
  Running seed 49...
  Extracting adult points from: merged_snapshots_20260908_092139.csv
  Collected 691,366 adult points
  KDE computed with bandwidth=0.1273
  ✅ Boyce Index: 0.935

[Run 2/30] Winter 2020
  Date: 2020-02-08T00:00:00 | Seed: 56
  Running seed 56...
  Extracting adult points from: merged_snapshots_20260908_095715.csv
  Collected 692,685 adult points
  KDE computed with bandwidth=0.1266
  ✅ Boyce Index: 0.888

[Run 3/30] Winter 2020
  Date: 2020-01-05T00:00:00 | Seed: 63
  Running seed 63...
  Extracting adult points from: merged_snapshots_20260908_103210.csv
  Collected 682,917 adult points
  KDE computed with bandwidth=0.1270
  ✅ Boyce Index: 0.942

[Run 4/30] Spring 2020
  Date: 2020-04-22T00:00:00 | Seed: 70
  Running seed 70...
  Extracting adult points from: merged_snapshots_20260908_111758.csv
  Collected 691,551 adult points
  KDE computed with ba

In [10]:
# ======================================================================
# 8. PROCESS SIMULATION FILES WITH DATETIME
# ======================================================================

def tick_to_datetime(tick_id, start_date, tick_minutes):
    """Convert a tick ID to a datetime object."""
    delta_minutes = int(tick_id) * tick_minutes
    return start_date + timedelta(minutes=delta_minutes)

def process_simulation_files_with_datetime(input_dir, output_dir, seasonal_runs):
    """Process all merged simulation files with datetime conversion."""
    os.makedirs(output_dir, exist_ok=True)
    
    # Find all merged files
    pattern = os.path.join(input_dir, "**", "merged_*.csv")
    files = glob.glob(pattern, recursive=True)
    
    if not files:
        print(f"No merged files found in {input_dir}")
        return []
    
    print(f"Found {len(files)} merged files to process")
    
    processed_files = []
    for filepath in files:
        basename = os.path.basename(filepath).replace('.csv', '_with_datetime.csv')
        output_path = os.path.join(output_dir, basename)
        
        # Find the corresponding start date from run metadata
        seed_dir = os.path.dirname(filepath)
        seed = int(os.path.basename(seed_dir).split('_')[1])
        
        # Find matching run info
        run_info = next((r for r in seasonal_runs if r['seed'] == seed), None)
        if run_info:
            start_date = datetime.fromisoformat(run_info['start_date'].replace('T', ' ').split('+')[0])
        else:
            start_date = datetime(2020, 1, 1, 0, 0, 0)
        
        # Process the file
        print(f"Converting: {os.path.basename(filepath)} (start: {start_date})")
        
        try:
            first_chunk = True
            total_rows = 0
            
            for chunk in pd.read_csv(filepath, chunksize=Config.CHUNKSIZE, low_memory=False):
                # Find TickCount column
                tick_col = None
                for col in chunk.columns:
                    if 'tick' in col.lower():
                        tick_col = col
                        break
                
                if tick_col is None:
                    print("  Warning: No TickCount column found!")
                    break
                
                chunk['DateTime'] = chunk[tick_col].apply(
                    lambda x: tick_to_datetime(x, start_date, Config.TICK_MINUTES).strftime("%Y-%m-%d %H:%M:%S")
                    if pd.notna(x) else ''
                )
                chunk['DateTimeISO'] = chunk[tick_col].apply(
                    lambda x: tick_to_datetime(x, start_date, Config.TICK_MINUTES).isoformat()
                    if pd.notna(x) else ''
                )
                
                mode = 'w' if first_chunk else 'a'
                header = first_chunk
                chunk.to_csv(output_path, mode=mode, header=header, index=False)
                
                total_rows += len(chunk)
                first_chunk = False
            
            print(f"  ✅ Processed {total_rows:,} rows")
            processed_files.append(output_path)
            
        except Exception as e:
            print(f"  Error: {e}")
    
    print(f"✅ Processed {len(processed_files)} files with datetime")
    return processed_files

if simulation_files:
    print("\n" + "="*60)
    print("PROCESSING SIMULATION FILES WITH DATETIME")
    print("="*60)
    
    processed_files = process_simulation_files_with_datetime(
        Config.SIMULATION_OUTPUT,
        Config.PROCESSED_OUTPUT,
        seasonal_runs
    )
    
    print(f"✅ Processed {len(processed_files)} files with datetime")


PROCESSING SIMULATION FILES WITH DATETIME
Found 30 merged files to process
Converting: merged_snapshots_20260908_135216.csv (start: 2020-07-18 00:00:00)
  ✅ Processed 1,144,428 rows
Converting: merged_snapshots_20260908_142059.csv (start: 2020-10-23 00:00:00)
  ✅ Processed 1,151,451 rows
Converting: merged_snapshots_20260908_150014.csv (start: 2020-11-08 00:00:00)
  ✅ Processed 1,153,643 rows
Converting: merged_snapshots_20260908_153745.csv (start: 2020-11-19 00:00:00)
  ✅ Processed 1,153,655 rows
Converting: merged_snapshots_20260908_160318.csv (start: 2021-02-26 00:00:00)
  ✅ Processed 1,151,085 rows
Converting: merged_snapshots_20260908_162744.csv (start: 2021-01-25 00:00:00)
  ✅ Processed 1,145,210 rows
Converting: merged_snapshots_20260908_165246.csv (start: 2021-01-23 00:00:00)
  ✅ Processed 1,148,788 rows
Converting: merged_snapshots_20260908_171311.csv (start: 2021-05-11 00:00:00)
  ✅ Processed 1,137,760 rows
Converting: merged_snapshots_20260908_173358.csv (start: 2021-05-05 

In [11]:
# ======================================================================
# 9. BOYCE INDEX ANALYSIS AND VISUALIZATION
# ======================================================================

if validation_results:
    df_results = pd.DataFrame(validation_results)
    
    # ======================================================================
    # Overall statistics
    # ======================================================================
    n = len(df_results)
    mean_bi = df_results['boyce_index'].mean()
    std_bi = df_results['boyce_index'].std()
    median_bi = df_results['boyce_index'].median()
    min_bi = df_results['boyce_index'].min()
    max_bi = df_results['boyce_index'].max()
    
    se = std_bi / np.sqrt(n)
    ci_low, ci_high = t.interval(0.95, n-1, loc=mean_bi, scale=se)
    
    print("\n" + "="*60)
    print("SPATIO-TEMPORAL BOYCE INDEX RESULTS")
    print("="*60)
    print(f"\nNumber of runs: {n}")
    print(f"Mean Boyce Index: {mean_bi:.3f} ± {std_bi:.3f}")
    print(f"Median: {median_bi:.3f}")
    print(f"95% CI: [{ci_low:.3f}, {ci_high:.3f}]")
    print(f"Range: {min_bi:.3f} - {max_bi:.3f}")
    
    # ======================================================================
    # Seasonal statistics
    # ======================================================================
    print("\n" + "-"*60)
    print("SEASONAL BOYCE INDEX RESULTS")
    print("-"*60)
    
    seasonal_stats = df_results.groupby('season').agg({
        'boyce_index': ['mean', 'std', 'count']
    }).round(3)
    
    for season in seasonal_stats.index:
        mean = seasonal_stats.loc[season, ('boyce_index', 'mean')]
        std = seasonal_stats.loc[season, ('boyce_index', 'std')]
        count = seasonal_stats.loc[season, ('boyce_index', 'count')]
        print(f"  {season.capitalize():8s}: {mean:.3f} ± {std:.3f} (n={count:.0f})")
    
    # ======================================================================
    # Save results
    # ======================================================================
    df_results.to_csv(os.path.join(Config.MONTE_CARLO_OUTPUT, "validation_results.csv"), index=False)
    
    # ======================================================================
    # FIGURES
    # ======================================================================
    
    # Figure 1: Overall Boyce Index Distribution
    fig, ax = plt.subplots(figsize=(10, 6))
    
    ax.hist(df_results['boyce_index'], bins=15, edgecolor='black', 
            alpha=0.7, color='steelblue')
    ax.axvline(mean_bi, color='red', linestyle='--', 
               linewidth=2, label=f'Mean: {mean_bi:.3f}')
    ax.axvline(median_bi, color='green', linestyle='--', 
               linewidth=2, label=f'Median: {median_bi:.3f}')
    ax.axvline(ci_low, color='red', linestyle=':', linewidth=1.5, 
               label=f'95% CI: [{ci_low:.3f}, {ci_high:.3f}]')
    ax.axvline(ci_high, color='red', linestyle=':', linewidth=1.5)
    ax.axvline(0.6, color='orange', linestyle='--', linewidth=1.5, 
               alpha=0.7, label='Good performance threshold')
    
    ax.set_xlabel('Boyce Index', fontsize=11)
    ax.set_ylabel('Frequency', fontsize=11)
    ax.set_title(f'Overall Boyce Index Distribution (n={n} runs)', fontsize=12)
    ax.legend(loc='upper left')
    ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(os.path.join(Config.FIGURES_OUTPUT, 'boyce_index_distribution.png'), 
                dpi=300, bbox_inches='tight')
    plt.close(fig)
    print(f"✓ Figure saved: {Config.FIGURES_OUTPUT}/boyce_index_distribution.png")
    
    # Figure 2: Seasonal Box Plot
    fig, ax = plt.subplots(figsize=(10, 6))
    
    # Prepare data for boxplot
    seasons_order = ['winter', 'spring', 'summer', 'autumn']
    season_data = []
    season_labels = []
    for season in seasons_order:
        data = df_results[df_results['season'] == season]['boyce_index'].values
        if len(data) > 0:
            season_data.append(data)
            season_labels.append(season.capitalize())
    
    bp = ax.boxplot(season_data, patch_artist=True, labels=season_labels)
    colors = ['#4A90D9', '#2ECC71', '#E74C3C', '#F39C12']
    for patch, color in zip(bp['boxes'], colors):
        patch.set_facecolor(color)
        patch.set_alpha(0.7)
    
    ax.set_ylabel('Boyce Index', fontsize=11)
    ax.set_xlabel('Season', fontsize=11)
    ax.set_title('Spatio-Temporal Boyce Index by Season', fontsize=12)
    ax.grid(True, alpha=0.3, axis='y')
    ax.axhline(0.6, color='orange', linestyle='--', linewidth=1.5, 
               alpha=0.7, label='Good performance threshold')
    ax.legend()
    
    plt.tight_layout()
    plt.savefig(os.path.join(Config.FIGURES_OUTPUT, 'boyce_index_seasonal_boxplot.png'), 
                dpi=300, bbox_inches='tight')
    plt.close(fig)
    print(f"✓ Figure saved: {Config.FIGURES_OUTPUT}/boyce_index_seasonal_boxplot.png")
    
    # Figure 3: Individual Run Results with Seasonal Coloring
    fig, ax = plt.subplots(figsize=(12, 6))
    
    color_map = {'winter': '#4A90D9', 'spring': '#2ECC71', 
                 'summer': '#E74C3C', 'autumn': '#F39C12'}
    
    x_pos = np.arange(len(df_results))
    for i, row in df_results.iterrows():
        color = color_map.get(row['season'], 'gray')
        ax.bar(i, row['boyce_index'], color=color, alpha=0.7, edgecolor='black', linewidth=0.5)
    
    ax.axhline(mean_bi, color='red', linestyle='--', linewidth=2,
               label=f'Overall Mean: {mean_bi:.3f}')
    ax.axhline(0.6, color='orange', linestyle='--', linewidth=1.5,
               alpha=0.7, label='Good performance threshold')
    
    ax.set_xlabel('Run Number', fontsize=11)
    ax.set_ylabel('Boyce Index', fontsize=11)
    ax.set_title('Individual Run Results (Colored by Season)', fontsize=12)
    
    # Create legend
    legend_elements = [
        plt.Rectangle((0,0), 1, 1, facecolor=color_map['winter'], label='Winter'),
        plt.Rectangle((0,0), 1, 1, facecolor=color_map['spring'], label='Spring'),
        plt.Rectangle((0,0), 1, 1, facecolor=color_map['summer'], label='Summer'),
        plt.Rectangle((0,0), 1, 1, facecolor=color_map['autumn'], label='Autumn'),
    ]
    ax.legend(handles=legend_elements, loc='upper left')
    ax.grid(True, alpha=0.3, axis='y')
    ax.set_xlim(-0.5, len(df_results) - 0.5)
    
    plt.tight_layout()
    plt.savefig(os.path.join(Config.FIGURES_OUTPUT, 'boyce_index_individual.png'), 
                dpi=300, bbox_inches='tight')
    plt.close(fig)
    print(f"✓ Figure saved: {Config.FIGURES_OUTPUT}/boyce_index_individual.png")
    
    # Figure 4: Boyce curve for best and worst runs
    best_run = df_results.loc[df_results['boyce_index'].idxmax()]
    worst_run = df_results.loc[df_results['boyce_index'].idxmin()]
    
    for run in [best_run, worst_run]:
        label = f"Best (BI={run['boyce_index']:.3f})" if run is best_run else f"Worst (BI={run['boyce_index']:.3f})"
        filepath = run['file']
        density, grid_x, grid_y = extract_adult_suitability(filepath)
        if density is not None:
            occ_points = list(zip(occurrences.geometry.x, occurrences.geometry.y))
            boyce_idx, thresholds, boyce_values = calculate_spatiotemporal_boyce_index(
                density, occ_points, grid_x, grid_y
            )
            if boyce_idx is not None:
                plot_spatiotemporal_boyce_index(boyce_idx, thresholds, boyce_values, 
                                               f"{label}", Config.FIGURES_OUTPUT)
    
    # Figure 5: Time series of Boyce Index
    fig, ax = plt.subplots(figsize=(12, 6))
    
    # Sort by date
    df_sorted = df_results.sort_values('start_date')
    
    # Create time series plot
    for season in ['winter', 'spring', 'summer', 'autumn']:
        mask = df_sorted['season'] == season
        if mask.any():
            ax.scatter(df_sorted[mask]['start_date'], 
                      df_sorted[mask]['boyce_index'],
                      label=season.capitalize(), 
                      alpha=0.7, s=50,
                      color=color_map[season])
    
    ax.axhline(mean_bi, color='red', linestyle='--', linewidth=2,
               label=f'Overall Mean: {mean_bi:.3f}')
    ax.axhline(0.6, color='orange', linestyle='--', linewidth=1.5,
               alpha=0.7, label='Good performance threshold')
    
    ax.set_xlabel('Simulation Start Date', fontsize=11)
    ax.set_ylabel('Boyce Index', fontsize=11)
    ax.set_title('Spatio-Temporal Boyce Index Over Time', fontsize=12)
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.xticks(rotation=45)
    
    plt.tight_layout()
    plt.savefig(os.path.join(Config.FIGURES_OUTPUT, 'boyce_index_timeseries.png'), 
                dpi=300, bbox_inches='tight')
    plt.close(fig)
    print(f"✓ Figure saved: {Config.FIGURES_OUTPUT}/boyce_index_timeseries.png")

else:
    print("❌ No validation results to analyze")


SPATIO-TEMPORAL BOYCE INDEX RESULTS

Number of runs: 30
Mean Boyce Index: 0.895 ± 0.099
Median: 0.921
95% CI: [0.858, 0.932]
Range: 0.415 - 0.961

------------------------------------------------------------
SEASONAL BOYCE INDEX RESULTS
------------------------------------------------------------
  Autumn  : 0.917 ± 0.047 (n=6)
  Spring  : 0.835 ± 0.190 (n=7)
  Summer  : 0.909 ± 0.024 (n=6)
  Winter  : 0.914 ± 0.048 (n=11)
✓ Figure saved: /mnt/monadworld/projects/phd/article_manuscripts/new/manuscript3/figs/case_study/figures/boyce_index_distribution.png
✓ Figure saved: /mnt/monadworld/projects/phd/article_manuscripts/new/manuscript3/figs/case_study/figures/boyce_index_seasonal_boxplot.png
✓ Figure saved: /mnt/monadworld/projects/phd/article_manuscripts/new/manuscript3/figs/case_study/figures/boyce_index_individual.png
  Extracting adult points from: merged_snapshots_20260908_223208.csv
  Collected 691,104 adult points
  KDE computed with bandwidth=0.1270
  ✓ Boyce Index plot saved to

In [12]:
# ======================================================================
# 10. PUBLICATION SUMMARY
# ======================================================================

print("\n" + "="*70)
print("PUBLICATION SUMMARY - Anopheles stephensi Case Study")
print("="*70)

if validation_results:
    print(f"\n📊 Spatio-Temporal Model Validation Summary:")
    print(f"  • Overall Boyce Index: {mean_bi:.3f} ± {std_bi:.3f} (n={n} runs)")
    print(f"  • 95% CI: [{ci_low:.3f}, {ci_high:.3f}]")
    
    if mean_bi > 0.8:
        print(f"  • Overall Performance: Excellent")
    elif mean_bi > 0.6:
        print(f"  • Overall Performance: Good")
    elif mean_bi > 0.4:
        print(f"  • Overall Performance: Moderate")
    else:
        print(f"  • Overall Performance: Poor")
    
    print(f"\n📈 Seasonal Performance:")
    for season in ['winter', 'spring', 'summer', 'autumn']:
        mask = df_results['season'] == season
        if mask.any():
            season_mean = df_results[mask]['boyce_index'].mean()
            season_std = df_results[mask]['boyce_index'].std()
            count = mask.sum()
            print(f"  • {season.capitalize():8s}: {season_mean:.3f} ± {season_std:.3f} (n={count})")

print(f"\n📁 Output Directory: {Config.OUTPUT_BASE}")
print(f"  - figures/ : All validation figures")
print(f"  - simulations/ : Individual simulation outputs")
print(f"  - processed/ : Files with datetime conversion")
print(f"  - monte_carlo/ : Validation results")
print(f"  - validation_report.txt : Complete report")

print("\n✅ Seasonal case study validation complete!")
print("="*70)


PUBLICATION SUMMARY - Anopheles stephensi Case Study

📊 Spatio-Temporal Model Validation Summary:
  • Overall Boyce Index: 0.895 ± 0.099 (n=30 runs)
  • 95% CI: [0.858, 0.932]
  • Overall Performance: Excellent

📈 Seasonal Performance:
  • Winter  : 0.914 ± 0.048 (n=11)
  • Spring  : 0.835 ± 0.190 (n=7)
  • Summer  : 0.909 ± 0.024 (n=6)
  • Autumn  : 0.917 ± 0.047 (n=6)

📁 Output Directory: /mnt/monadworld/projects/phd/article_manuscripts/new/manuscript3/figs/case_study
  - figures/ : All validation figures
  - simulations/ : Individual simulation outputs
  - processed/ : Files with datetime conversion
  - monte_carlo/ : Validation results
  - validation_report.txt : Complete report

✅ Seasonal case study validation complete!


In [13]:
# ======================================================================
# 11. COMPLETE REPORT
# ======================================================================

if validation_results:
    with open(os.path.join(Config.OUTPUT_BASE, 'validation_report.txt'), 'w') as f:
        f.write("="*70 + "\n")
        f.write("Anopheles stephensi SEASONAL VALIDATION REPORT\n")
        f.write("="*70 + "\n\n")
        f.write(f"Date: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
        f.write(f"Total runs: {n}\n")
        f.write(f"Simulation duration per run: {Config.SIMULATION_DAYS} days\n")
        f.write(f"Seeding: Full study site (not occurrence-based)\n")
        f.write(f"Occurrence data: {Config.OCCURRENCE_YEAR_START}-{Config.OCCURRENCE_YEAR_END}\n\n")
        
        f.write("SPATIO-TEMPORAL BOYCE INDEX RESULTS\n")
        f.write("-"*40 + "\n")
        f.write(f"  Mean: {mean_bi:.4f} ± {std_bi:.4f}\n")
        f.write(f"  Median: {median_bi:.4f}\n")
        f.write(f"  95% CI: [{ci_low:.4f}, {ci_high:.4f}]\n")
        f.write(f"  Min: {min_bi:.4f}\n")
        f.write(f"  Max: {max_bi:.4f}\n\n")
        
        f.write("SEASONAL BOYCE INDEX RESULTS\n")
        f.write("-"*40 + "\n")
        for season in ['winter', 'spring', 'summer', 'autumn']:
            mask = df_results['season'] == season
            if mask.any():
                s_mean = df_results[mask]['boyce_index'].mean()
                s_std = df_results[mask]['boyce_index'].std()
                s_count = mask.sum()
                f.write(f"  {season.capitalize():8s}: {s_mean:.4f} ± {s_std:.4f} (n={s_count})\n")
        f.write("\n")
        
        f.write("INTERPRETATION\n")
        f.write("-"*40 + "\n")
        if mean_bi > 0.8:
            f.write("  Excellent model performance\n")
        elif mean_bi > 0.6:
            f.write("  Good model performance\n")
        elif mean_bi > 0.4:
            f.write("  Moderate model performance\n")
        else:
            f.write("  Poor model performance\n")
        f.write(f"  The model successfully predicts {mean_bi*100:.1f}% of the occurrence pattern\n\n")
        
        f.write("MODEL SETUP\n")
        f.write("-"*40 + "\n")
        f.write(f"  Species: Anopheles stephensi\n")
        f.write(f"  Study area: Somali region, Ethiopia\n")
        f.write(f"  Parameters: Fitted from literature (Komi et al. 2025)\n")
        f.write(f"  Simulation engine: Agent-based model\n")
        f.write(f"  Time resolution: 15-minute ticks\n")
        f.write(f"  Spatial resolution: 0.001 degrees\n\n")
        
        f.write("VALIDATION DATA\n")
        f.write("-"*40 + "\n")
        f.write(f"  Total occurrence points: {len(occurrences)}\n")
        f.write(f"  Year range: {Config.OCCURRENCE_YEAR_START}-{Config.OCCURRENCE_YEAR_END}\n")
        f.write(f"  Data source: Ethiopian occurrence records\n\n")
        
        f.write("="*70 + "\n")
        f.write("END OF REPORT\n")
    
    print(f"✓ Complete report saved to: {Config.OUTPUT_BASE}/validation_report.txt")

else:
    print("❌ No validation results to analyze")

✓ Complete report saved to: /mnt/monadworld/projects/phd/article_manuscripts/new/manuscript3/figs/case_study/validation_report.txt


In [14]:
# ======================================================================
# 12. FINAL CLEANUP AND SUMMARY
# ======================================================================

print("\n" + "="*70)
print("FINAL SUMMARY")
print("="*70)

print(f"\n✅ All processing complete!")
print(f"\nOutput directory: {Config.OUTPUT_BASE}")
print(f"\nDirectory contents:")

for dirname in ['figures', 'simulations', 'processed', 'monte_carlo']:
    dirpath = os.path.join(Config.OUTPUT_BASE, dirname)
    if os.path.exists(dirpath):
        files = os.listdir(dirpath)
        print(f"  {dirname}/: {len(files)} files")

print("\n📄 Key output files:")
print(f"  - validation_report.txt : Complete validation report")
print(f"  - figures/boyce_index_distribution.png : Overall distribution")
print(f"  - figures/boyce_index_seasonal_boxplot.png : Seasonal comparison")
print(f"  - figures/boyce_index_timeseries.png : Time series")
print(f"  - monte_carlo/validation_results.csv : Individual run results")

print("\n" + "="*70)
print("✅ SEASONAL CASE STUDY COMPLETE")
print("="*70)


FINAL SUMMARY

✅ All processing complete!

Output directory: /mnt/monadworld/projects/phd/article_manuscripts/new/manuscript3/figs/case_study

Directory contents:
  figures/: 6 files
  simulations/: 30 files
  processed/: 31 files
  monte_carlo/: 2 files

📄 Key output files:
  - validation_report.txt : Complete validation report
  - figures/boyce_index_distribution.png : Overall distribution
  - figures/boyce_index_seasonal_boxplot.png : Seasonal comparison
  - figures/boyce_index_timeseries.png : Time series
  - monte_carlo/validation_results.csv : Individual run results

✅ SEASONAL CASE STUDY COMPLETE
